In [15]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = os.getenv('CLAUDE_MODEL')

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [25]:
dataset = generate_dataset()
dataset

[{'task': "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
  'format': 'regex'},
 {'task': 'Create a JSON configuration object for an AWS Lambda function with environment variables, timeout, and memory allocation settings',
  'format': 'json'},
 {'task': 'Write a Python function that converts an AWS CloudWatch timestamp (Unix epoch milliseconds) to a human-readable date string',
  'format': 'python'}]

In [26]:
with open('dataset.json', 'w') as file:
    json.dump(dataset, file, indent=2)

In [34]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```")
    output = chat(messages, stop_sequences=["```"])
    return output

In [18]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [28]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [29]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [13]:
with open('dataset.json', 'r') as file:
    dataset = json.load(file)

results = run_eval(dataset)

In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The S3 bucket name to validate\n        \n    Returns:\n        bool: True if the bucket name is valid, False otherwise\n    \"\"\"",
    "test_case": {
      "task": "Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase letters, numbers, hyphens only, 3-63 characters)"
    },
    "score": 10
  },
  {
    "output

In [20]:
with open('dataset.json', 'r') as file:
    dataset = json.load(file)

results = run_eval(dataset)

In [21]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Naming Convention Validator\n\n```python\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 bucket naming rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters (a-z), numbers (0-9), and hyphens (-)\n    - Cannot start or end with a hyphen\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The S3 bucket name to validate\n        \n    Returns:\n        bool: True if the bucket name is valid, False otherwise\n    \"\"\"\n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length constraint (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Check if it starts or ends with a hyphen\n    i

In [30]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format {format}")

In [31]:
with open('dataset.json', 'r') as file:
    dataset = json.load(file)

results = run_eval(dataset)

Average score: 6.333333333333333


In [32]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n\n# Test\nprint(extract_s3_bucket('s3://my-bucket/path/to/file.txt'))\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.0,
    "reasoning": "The solution correctly solves the basic task with a simple regex pattern. However, it lacks production-readiness due to missing input validation, case-sensitivity issues, and no bucket name validation against AWS naming rules. The test case works but doesn't cover edge cases like malformed URIs, uppercase variants, or invalid bucket names. Adding input validation, case-insensitive matching, and basic bucket name validation would significantly improve robustness."
  },
  {
    "output": "\nimport json\n\nlambda_config = {\n    \"FunctionNa

In [35]:
with open('dataset.json', 'r') as file:
    dataset = json.load(file)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 6.166666666666667
[
  {
    "output": "python\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([a-z0-9.-]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 7.5,
    "reasoning": "The solution works for basic lowercase S3 URIs but has critical limitations in real-world usage. AWS bucket naming rules allow uppercase letters (though typically normalized to lowercase), and the function should either validate input or use boto3's built-in parsing. The regex pattern needs to be updated to `s3://([a-zA-Z0-9._-]+)` at minimum, and ideally the solution should handle edge cases like empty URIs, invalid formats, or S3A schemes. Consider using `urllib.parse.urlparse()` or boto3's URI parsing for a more robust, maintainable approach."
  },
  {
    "output": 